<a href="https://colab.research.google.com/github/ksuplee/AI_Agent/blob/main/09_3_MCP_based_API_DB_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[실습 09-3] MCP 기반 외부 API·DB 연동 실습: 통합형 에이전트 구축  

### 실습목표

- MCP Server를 시뮬레이션하여 AI 에이전트가 외부 데이터베이스(DB)와 실시간 API에 접근하도록 구성할 수 있다.  

- **Resource(정적 컨텍스트)** 와 **Tool(동적 행동)** 을 결합하여 복합적인 문제를 해결하는 통합형 에이전트를 구현한다.  

- 모델에 종속되지 않는 표준화된 데이터 접근 아키텍처를 통해 에이전트-도구-데이터의 **느슨한 결합(Loose Coupling)** 을 체험한다.  

1. 환경 준비 및 라이브러리 설치  

- 실습을 위해 필요한 LangChain 및 Google Gemini 라이브러리를 설정합니다.  

In [ ]:
# 1. 필수 라이브러리 설치
!pip install -q -U langchain langchain-community langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.4/719.4 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.5/236.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-a

In [ ]:
# Google API Key 설정
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

print("Gemini API 설정 완료")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Gemini API 설정 완료


In [ ]:
# 1. 필수 라이브러리 설치
# !pip install -q -U langchain langchain-community langchain-google-genai

import google.generativeai as genai
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain.tools import tool

# Gemini API 설정
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model='gemini-flash-latest', api_key=GOOGLE_API_KEY)

2. [실습 1] MCP Server 시뮬레이션 (DB Resource & API Tool)  

- 에이전트 코드와 분리된 MCP Server 역할을 정의합니다. 서버는 DB 조회 로직과 API 호출 방식을 캡슐화하여 제공합니다.  

In [ ]:
# --- [MCP Server 영역] --- [cite: 09-3-1]

# 1. Resource: 사내 지식 베이스 (SQLite DB 시뮬레이션) [cite: 09-3-2]
# 모델이 배경 지식으로 참고할 수 있는 정적 데이터입니다.
MCP_DATABASE = {
    "user_manual": """
    [사내 복장 규정 및 매뉴얼]
    - 기온 20도 이상: 비즈니스 캐주얼 (가벼운 셔츠, 면바지) 추천.
    - 기온 10도 ~ 20도: 가벼운 외투(자켓, 가디건) 착용 권장.
    - 기온 10도 미만: 코트 또는 패딩 등 보온 의류 필수.
    - 특이사항: 비가 올 경우를 대비해 항상 사무실에 우산을 비치하십시오.
    """
}

# 2. Tool: 실시간 외부 API (날씨 검색 시뮬레이션) [cite: 09-3-3]
# 필요할 때만 실행하는 동적 기능입니다.
@tool
def search_weather_api(city: str) -> str:
    """특정 도시의 실시간 날씨와 기온 정보를 검색합니다."""
    # 실제 환경에서는 외부 날씨 API(OpenWeather 등)를 호출합니다.
    simulated_weather = {
        "서울": "기온 8도, 날씨 흐림",
        "수원": "기온 12도, 날씨 비",
        "제주": "기온 18도, 날씨 맑음"
    }
    return simulated_weather.get(city, "날씨 정보를 찾을 수 없습니다.")

mcp_tools = [search_weather_api]

3. [실습 2] MCP 기반 통합형 에이전트 구현  

- **Resource(읽기)** 와 **Tool(실행)** 을 결합하여 사용자의 복합적인 요청을 처리하는 에이전트 로직을 구성합니다 [cite: 09-3-3].  

In [ ]:
# --- [MCP Host / Agent 영역] --- [cite: 09-3-1]

# 도구 바인딩
llm_with_tools = llm.bind_tools(mcp_tools)

def run_mcp_integrated_agent(user_query: str):
    # 1. Discovery & Resource Injection [cite: 09-3-2]
    # 에이전트는 DB 연결 로직을 직접 포함하지 않고 리소스를 읽어옵니다.
    db_context = MCP_DATABASE["user_manual"]

    # 2. Contextual Prompt 설계 [cite: 09-3-4]
    prompt = ChatPromptTemplate.from_messages([
        ("system", f"너는 사내 매뉴얼(Resource)을 숙지한 인공지능 비서야. 아래 정보를 참고해 답변해줘.\n\n[사내 매뉴얼]\n{db_context}"),
        ("human", "{input}")
    ])

    # 3. 1차 추론 및 도구 선택 [cite: 09-3-3]
    chain = prompt | llm_with_tools
    ai_msg = chain.invoke({"input": user_query})

    # 4. Tool 실행 (실시간 데이터 연계) [cite: 09-3-3]
    if ai_msg.tool_calls:
        tool_call = ai_msg.tool_calls[0]
        print(f"[*] Tool 호출 중: {tool_call['name']}({tool_call['args']})")

        observation = search_weather_api.invoke(tool_call["args"])
        print(f"[*] Observation 수집: {observation}")

        # 5. 최종 통합 추론 (정적 지식 + 동적 정보) [cite: 09-3-3]
        final_response = llm_with_tools.invoke([
            HumanMessage(content=user_query),
            ai_msg,
            ToolMessage(content=observation, tool_call_id=tool_call["id"])
        ])
        return final_response.content

    return ai_msg.content

print("✅ 통합형 MCP 에이전트가 준비되었습니다.")

✅ 통합형 MCP 에이전트가 준비되었습니다.


4. 시나리오 테스트 및 검증  

- 에이전트가 지식(매뉴얼)과 행동(실시간 검색)을 어떻게 결합하는지 확인합니다 [cite: 09-3-3].  

In [ ]:
# 시나리오: DB의 매뉴얼과 실시간 날씨를 결합하여 추천 [cite: 09-3-3]
test_query = "서울 사무실로 출근하려고 하는데, 오늘 날씨에 맞춰서 어떻게 입고 가면 좋을까?"

print(f"Q: {test_query}")
response = run_mcp_integrated_agent(test_query)
print("-" * 50)
print(f"A: {response}")

Q: 서울 사무실로 출근하려고 하는데, 오늘 날씨에 맞춰서 어떻게 입고 가면 좋을까?
[*] Tool 호출 중: search_weather_api({'city': '서울'})
[*] Observation 수집: 기온 8도, 날씨 흐림
--------------------------------------------------
A: [{'type': 'text', 'text': '오늘 서울의 기온은 **8도**로 쌀쌀하고 날씨는 흐립니다.\n\n**[사내 복장 규정 및 매뉴얼]**에 따라 10°C 미만의 기온이므로, **코트 또는 패딩 등 보온 의류가 필수**입니다.\n\n따라서 다음과 같이 입고 가시는 것을 추천합니다.\n\n1.  **외투:** 따뜻한 **두꺼운 코트**나 **패딩**을 착용하여 체온을 유지해야 합니다.\n2.  **내부:** 외투 안에 **따뜻한 니트**나 **기모 처리된 바지**를 입으시면 좋습니다. 사무실 내에서는 비즈니스 캐주얼(셔츠, 블라우스, 면바지 등)을 유지해 주세요.\n3.  **액세서리:** 8도면 체감 온도가 더 낮을 수 있으니, **목도리**나 **장갑** 같은 방한 액세서리를 챙기시는 것도 추천합니다.\n\n쌀쌀한 날씨에 감기 걸리지 않도록 따뜻하게 입고 출근하세요!', 'extras': {'signature': 'CosLAXLI2nwjy+6Bu4XN425qFbBm6duqJCvPfLfLtapP5I/Hf+uWMV9JBRPMVSnVV35Rw8B+kms24/Wp2YETfSglKToIKUuuoQnmMNUUtBr3Pl8hlPDz/DT8UaOKNtcsLV6K0ktMAsgfFgdGp6+9ShwA16v1LWsAq1kjsKt4VY3POxbNd8VowWXMKck0ZKs5mjJI04j45FB6iWsR+gtrlSJbaD2+LKCF0Mz5O/wXZYo0LUINqUMliEZktp2PFfKWxiASrTPozObTvAmbCnCWYDBKRw/7ZVt4vu2KpxBgeATiEZxjS9+ZwCLrS4Asoh7pP0Lm304RK0FsH9o4EDoUNUsOa

- 관찰 포인트  
    - 느슨한 결합: 에이전트 코드에는 sqlite3.connect()나 API 인증 키가 보이지 않습니다. 모든 접근 로직은 MCP Server 단에서 중앙 관리됩니다 [cite: 09-3, 09-3-4].  
    - 모델 독립성: 내부 모델을 GPT에서 Claude로 교체하더라도, 동일한 mcp://sqlite/user_manual 리소스 주소와 search_weather_api 도구 규격만 유지하면 연동 코드는 수정되지 않습니다.  
    - 보안 강화: 민감한 DB 계정 정보나 API Key는 에이전트 환경이 아닌 독립된 MCP Server 단에서만 노출되므로 보안 책임이 중앙화됩니다 [cite: 09-3-4].  

**실습점검**  

- Q1. 실시간 날씨 정보를 가져올 때 왜 Resource가 아닌 Tool을 사용했나요?  
    - A. Resource는 모델이 읽기 전용으로 참고하는 정적 데이터인 반면, 실시간 정보는 외부 API를 통해 동적으로 호출해야 하는 행동이기 때문에 Tool을 사용합니다.  

- Q2. 사내 매뉴얼이 업데이트되어도 에이전트 코드를 수정할 필요가 없는 이유는 무엇인가요?  
    - A. 에이전트는 데이터의 실제 구현을 모른 채 MCP Server가 제공하는 Resource 경로만 참조하고 있기 때문입니다 [cite: 09-3-1, 09-3-2].  